- SET TBLPROPERTIES
- delta.columnMapping.mode = 'name'
- Adding and dropping a column
- Checking the schema with DESCRIBE TABLE
2. Databricks VARIANT and JSON parsing
- parse_json() in PySpark
- Boolean values (true / false)
- Extracting nested fields with metadata:channel and metadata:device
- Querying a temporary view with SQL


In [0]:
from pyspark.sql.functions import col

In [0]:
%sql 
alter table workspace.default.cars_jhansi_data add column metadata variant;
describe table workspace.default.cars_jhansi_data;


col_name,data_type,comment
make,string,null
bore,double,null
price,int,null
width,double,null
city-mpg,int,null
price_in_thousands,double,null
metadata,variant,null


In [0]:
%sql
rollback;

In [0]:
%sql
-- Enable column mapping first (required for DROP COLUMN)
ALTER TABLE workspace.default.cars_jhansi_data
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

-- Now drop the column
ALTER TABLE workspace.default.cars_jhansi_data DROP COLUMN metadata;

-- Verify the schema
DESCRIBE TABLE workspace.default.cars_jhansi_data;

col_name,data_type,comment
make,string,null
bore,double,null
price,int,null
width,double,null
city-mpg,int,null
price_in_thousands,double,null


In [0]:
%sql
ALTER TABLE workspace.default.cars_jhansi_data
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
);

In [0]:
%sql 
alter table workspace.default.trialtable add column metadata variant;
describe table workspace.default.trialtable;

col_name,data_type,comment
id,bigint,null
name,string,null
marks,bigint,null
metadata,variant,null


In [0]:
%sql
select * from trialtable;

id,name,marks,metadata
1,jhansi,88,null
2,Jhashvin,100,null
3,satz,89,null


In [0]:
from pyspark.sql.functions import col, parse_json


metadata_rows=[("1",'{"channel":"online", "device":"mobile","promo_applied":true}'),
               ("2",'{"channel":"offline", "device":"tablet","promo_applied"}')
               ]

df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
df_meta_variant = (df_meta.withColumn("metadata",parse_json(col("metadata"))))
display(df_meta_variant)
df_meta_variant.createOrReplaceTempView("metadata_updates")
print("metadata_updates registered")


---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-5193728150761254>, line 10
      8 df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
      9 df_meta_variant = (df_meta.withColumn("metadata",parse_json(col("metadata"))))
---> 10 display(df_meta_variant)
     11 df_meta_variant.createOrReplaceTempView("metadata_updates")
     12 print("metadata_updates registered")

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:115, in Display.display_connect_table(self, df, **kwargs)
    112     self.cf_helper.display_str

In [0]:
from pyspark.sql.functions import col, parse_json


metadata_rows=[("1",'{"channel":"online", "device":"mobile","promo_applied":true}'),
               ("2",'{"channel":"offline", "device":"tablet","promo_applied":jhansi}')
               ]

df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
df_meta_variant = (df_meta.withColumn("metadata",parse_json(col("metadata"))))
display(df_meta_variant)
df_meta_variant.createOrReplaceTempView("metadata_updates")
print("metadata_updates registered")

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-7421745915607333>, line 10
      8 df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
      9 df_meta_variant = (df_meta.withColumn("metadata",parse_json(col("metadata"))))
---> 10 display(df_meta_variant)
     11 df_meta_variant.createOrReplaceTempView("metadata_updates")
     12 print("metadata_updates registered")

File /databricks/python_shell/lib/dbruntime/display.py:185, in Display.display(self, input, *args, **kwargs)
    183     pass
    184 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 185     self.display_connect_table(input, **kwargs)
    186 elif isinstance(input, ConnectDataFrame):
    187     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:115, in Display.display_connect_table(self, df, **kwargs)
    112     self.cf_helper.display_str

In [0]:
from pyspark.sql.functions import col, parse_json


metadata_rows=[("1",'{"channel":"online", "device":"mobile","promo_applied":true}'),
               ("2",'{"channel":"offline", "device":"tablet","promo_applied":false}')
               ]

df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
df_meta_variant = (df_meta.withColumn("metadata",parse_json(col("metadata"))))
display(df_meta_variant)
df_meta_variant.createOrReplaceTempView("metadata_updates")
print("metadata_updates registered")

id,metadata
1,"{""channel"":""online"",""device"":""mobile"",""promo_applied"":true}"
2,"{""channel"":""offline"",""device"":""tablet"",""promo_applied"":false}"


metadata_updates registered


In [0]:
from pyspark.sql.functions import col, try_parse_json


metadata_rows=[("1",'{"channel":"online", "device":"mobile","promo_applied":true}'),
               ("2",'{"channel":"offline", "device":"tablet","promo_applied"}')
               ]

df_meta = spark.createDataFrame(metadata_rows,["id","metadata"])
df_meta_variant = (df_meta.withColumn("metadata",try_parse_json(col("metadata"))))
display(df_meta_variant)
df_meta_variant.createOrReplaceTempView("metadata_updates")
print("metadata_updates registered")

id,metadata
1,"{""channel"":""online"",""device"":""mobile"",""promo_applied"":true}"
2,null


metadata_updates registered


In [0]:
%sql
select id,metadata:channel as channel, metadata:device as device,metadata:promo_applied as prom from metadata_updates

id,channel,device,prom
1,"""online""","""mobile""",true
2,null,null,null


In [0]:
df_meta.show()

+---+--------------------+
| id|            metadata|
+---+--------------------+
|  1|{"channel":"onlin...|
|  2|{"channel":"offli...|
+---+--------------------+

